<a href="https://colab.research.google.com/github/kmaxat/ai-job-search/blob/master/BuildTuberAutoEditv2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BuildTuber Auto Edit v2.0

Turn your images, video clips and voiceover into a finished video automatically,
with accurate scene timing, Ken Burns motion and smooth transitions.

Everything runs inside your own Google Colab session.

### How to use

**Step 1:** Run the setup cell and wait about 30 seconds.

**Step 2:** Run the studio cell. The editor will open below.

Upload:

- your images and/or video clips
- your voiceover
- your script

Your script should contain one line per image or clip, in the same order as your
files.

Then click **Build Video**.

### Keep this tab open

Keep the Colab tab open and your computer awake while the video is rendering.

Longer videos may take several minutes.

In [ ]:
#@title Step 1 - Set up  { display-mode: "form" }
#@markdown Takes about 30 seconds. You only need to run this once.

#@markdown Tick this only if you want to load your files from a Google Drive
#@markdown folder instead of uploading them. It asks for read-only permission.
use_google_drive = False  #@param {type:"boolean"}

import subprocess, sys, time

t0 = time.time()
print('Installing... (about 30 seconds)')

# Pin the compatibility-critical packages used by Auto Edit.
#
# Gradio is pinned because this notebook depends on specific UI behaviour.
# faster-whisper is pinned because transcription and word timestamps are part
# of the sync pipeline.
# PyAV is pinned because faster-whisper 1.2.1 currently relies on an API that
# changed in PyAV 19, causing:
#
# TypeError: open() got an unexpected keyword argument 'metadata_errors'
#
# PyAV 18.1.0 is the compatible version for this build.
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'gradio==6.28.0',
        'faster-whisper==1.2.1',
        'av==18.1.0',
    ],
    check=True
)

# Checked rather than installed: both ship with Colab, and a missing one here is
# a much clearer message than the traceback it would cause later.
import shutil

assert shutil.which('ffmpeg'), 'ffmpeg is missing from this runtime.'

from PIL import Image  # noqa: F401

# Optional Google Drive connection.
DRIVE = None

if use_google_drive:
    print('Asking for permission to read your Drive folder...')

    from google.colab import auth
    from googleapiclient.discovery import build as _gbuild

    auth.authenticate_user()

    DRIVE = _gbuild(
        'drive',
        'v3',
        cache_discovery=False
    )

    print('Drive connected.')

# Import after installation so the versions shown below are the versions this
# runtime will actually use.
import gradio
import av
import faster_whisper

print(
    f'Ready in {time.time() - t0:.0f}s. '
    f'Gradio {gradio.__version__}, '
    f'faster-whisper {faster_whisper.__version__}, '
    f'PyAV {av.__version__}.'
)

print('Now run Step 2.')

In [ ]:
#@title Step 2 - Open the studio  { display-mode: "form" }
#@markdown Run this and the studio appears below. Leave this cell running while
#@markdown you use it.
#@markdown
#@markdown You may see a temporary gradio.live link when the studio starts.
#@markdown You can ignore it.

import os

WORK = '/content/btstudio'
TMP = f'{WORK}/_tmp'

for d in (WORK, TMP):
    os.makedirs(d, exist_ok=True)

# Read by gradio at import time, so it has to be set before the import below.
os.environ['GRADIO_TEMP_DIR'] = TMP

import gradio as gr
import difflib
import html
import json
import math
import re
import shutil
import subprocess
import time
import traceback
import zipfile

from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageOps

FPS = 30

SIZES = {
    '1920 x 1080  (landscape / YouTube)': (1920, 1080),
    '1080 x 1920  (vertical / Shorts)': (1080, 1920),
    '1080 x 1080  (square)': (1080, 1080),
}

IMAGE_EXT = (
    '.jpg',
    '.jpeg',
    '.png',
    '.webp',
    '.bmp'
)

VIDEO_EXT = (
    '.mp4',
    '.mov',
    '.m4v',
    '.webm',
    '.mkv',
    '.avi'
)

MEDIA_EXT = IMAGE_EXT + VIDEO_EXT


def is_video(path):
    return path.lower().endswith(VIDEO_EXT)


def run(cmd):
    p = subprocess.run(cmd, capture_output=True)

    if p.returncode != 0:
        raise gr.Error(
            'ffmpeg failed: ' +
            p.stderr.decode('utf8', 'ignore')[-400:]
        )

    return p


def duration(path):
    p = subprocess.run(
        [
            'ffprobe',
            '-v', 'error',
            '-show_entries', 'format=duration',
            '-of', 'csv=p=0',
            path
        ],
        capture_output=True
    )

    try:
        return float(p.stdout.decode().strip())

    except ValueError:
        raise gr.Error(
            'That audio file could not be read.'
        )


def has_sound(path):
    """Whether there is an actual audio stream in there."""

    p = subprocess.run(
        [
            'ffprobe',
            '-v', 'error',
            '-select_streams', 'a:0',
            '-show_entries', 'stream=codec_name',
            '-of', 'csv=p=0',
            path
        ],
        capture_output=True
    )

    return bool(p.stdout.decode().strip())


# ----------------------------------------------------------------- script

def read_script_file(path):
    """.txt or .docx."""

    if path.lower().endswith('.docx'):

        with zipfile.ZipFile(path) as z:
            xml = z.read(
                'word/document.xml'
            ).decode(
                'utf8',
                'ignore'
            )

        xml = re.sub(
            r'<w:(?:p|br|cr)(?=[ />])',
            '\n<w:x',
            xml
        )

        xml = re.sub(
            r'<[^>]+>',
            '',
            xml
        )

        return html.unescape(xml)

    with open(
        path,
        'r',
        encoding='utf8',
        errors='ignore'
    ) as f:

        return f.read()


def script_lines(text):
    """One scene per non-empty line."""

    return [
        ln.strip()
        for ln in text.replace(
            '\r\n',
            '\n'
        ).split('\n')
        if ln.strip()
    ]


def script_blocks(text):
    """Blank-line separated groups."""

    return [
        b.strip()
        for b in re.split(
            r'\n\s*\n',
            text.strip()
        )
        if b.strip()
    ]


def script_vs_files(text, paths):
    """Resolve script into scenes and compare with file count."""

    paths = list(paths or [])
    text = text or ''

    lines = script_lines(text)

    if text.strip() and len(lines) != len(paths):

        alt = script_blocks(text)

        if len(alt) == len(paths):
            lines = alt

    if not paths or not lines or len(lines) == len(paths):
        return lines, None

    n_vid = sum(
        is_video(p)
        for p in paths
    )

    things = (
        'images and clips'
        if n_vid
        else 'images'
    )

    thing = (
        'image or clip'
        if n_vid
        else 'image'
    )

    over = len(lines) - len(paths)

    if over > 0:

        detail = (
            'so there %s %d line%s too many, or one is missing. '
            'A line is any paragraph with something in it, so a '
            'paragraph holding one stray character counts even though '
            'it looks blank.'
            % (
                'is' if over == 1 else 'are',
                over,
                '' if over == 1 else 's'
            )
        )

    else:

        detail = (
            'so there %s %d too many, or lines that belong on '
            'their own are sharing a paragraph. Check for files '
            'that are not shots - a thumbnail or a logo is counted '
            'like any other %s.'
            % (
                'is' if over == -1 else 'are',
                -over,
                thing
            )
        )

    return (
        lines,
        (
            'Your script has %d lines but there are %d %s - %s'
            % (
                len(lines),
                len(paths),
                things,
                detail
            )
        )
    )


def bare_words(text):
    """Lower-case words, punctuation stripped."""

    return re.sub(
        r"[^a-z0-9' ]+",
        ' ',
        text.lower()
    ).split()


# ------------------------------------------------------------ drive input

DRIVE = globals().get('DRIVE')

AUDIO_EXT = (
    '.mp3',
    '.wav',
    '.m4a',
    '.aac',
    '.flac',
    '.ogg',
    '.opus'
)

TEXT_EXT = (
    '.txt',
    '.md',
    '.text',
    '.docx'
)

GDOC_MIME = (
    'application/vnd.google-apps.document'
)


def natural_key(name):
    """Sort 2.png before 10.png."""

    return [
        int(t)
        if t.isdigit()
        else t.lower()
        for t in re.split(
            r'(\d+)',
            name
        )
    ]


def drive_pull(
    link,
    progress=gr.Progress()
):

    if DRIVE is None:

        raise gr.Error(
            'Drive is not connected. Go back to Step 1, tick '
            '"use_google_drive", and run it again - it will ask for '
            'read-only permission.'
        )

    m = re.search(
        r'[-\w]{25,}',
        link or ''
    )

    if not m:

        raise gr.Error(
            'That does not look like a Drive folder link. Right-click the '
            'folder in Drive, choose Share, then Copy link, and paste the '
            'whole thing.'
        )

    folder_id = m.group(0)

    import io
    from googleapiclient.http import MediaIoBaseDownload

    progress(
        0.05,
        desc='Reading the folder...'
    )

    files = []
    token = None

    while True:

        res = DRIVE.files().list(
            q="'%s' in parents and trashed=false" % folder_id,
            fields='nextPageToken, files(id,name,mimeType)',
            pageSize=1000,
            pageToken=token
        ).execute()

        files.extend(
            res.get(
                'files',
                []
            )
        )

        token = res.get(
            'nextPageToken'
        )

        if not token:
            break

    dest_dir = f'{WORK}/drive'

    shutil.rmtree(
        dest_dir,
        ignore_errors=True
    )

    os.makedirs(
        dest_dir,
        exist_ok=True
    )

    def fetch(
        f,
        dest,
        export=None
    ):

        req = (
            DRIVE.files().export_media(
                fileId=f['id'],
                mimeType=export
            )
            if export
            else DRIVE.files().get_media(
                fileId=f['id']
            )
        )

        with io.FileIO(
            dest,
            'wb'
        ) as fh:

            dl = MediaIoBaseDownload(
                fh,
                req,
                chunksize=8 * 1024 * 1024
            )

            done = False

            while not done:
                _, done = dl.next_chunk()

    wanted = []

    for f in files:

        low = f['name'].lower()

        if low.endswith(MEDIA_EXT):

            wanted.append(
                (
                    f,
                    'media',
                    None
                )
            )

        elif low.endswith(AUDIO_EXT):

            wanted.append(
                (
                    f,
                    'audio',
                    None
                )
            )

        elif low.endswith(TEXT_EXT):

            wanted.append(
                (
                    f,
                    'text',
                    None
                )
            )

        elif f.get('mimeType') == GDOC_MIME:

            wanted.append(
                (
                    f,
                    'text',
                    'text/plain'
                )
            )

    if not wanted:

        raise gr.Error(
            'That folder has nothing usable in it. It needs your images or '
            'clips, one voiceover file, and your script.'
        )

    media = []
    audio = None
    script_text = ''

    for k, (
        f,
        kind,
        export
    ) in enumerate(wanted):

        progress(
            0.1 + 0.85 * k / len(wanted),
            desc=(
                'Downloading %d of %d - %s'
                % (
                    k + 1,
                    len(wanted),
                    f['name'][:40]
                )
            )
        )

        name = re.sub(
            r'[^\w.\- ]+',
            '_',
            f['name']
        )

        if export:
            name += '.txt'

        dest = os.path.join(
            dest_dir,
            name
        )

        fetch(
            f,
            dest,
            export
        )

        if kind == 'media':

            media.append(dest)

        elif kind == 'audio' and audio is None:

            audio = dest

        elif (
            kind == 'text'
            and not script_text.strip()
        ):

            script_text = read_script_file(
                dest
            )

    media.sort(
        key=lambda p:
        natural_key(
            os.path.basename(p)
        )
    )

    if not media:

        raise gr.Error(
            'No images or clips in that folder.'
        )

    if audio is None:

        raise gr.Error(
            'No voiceover in that folder. It needs one audio file (%s).'
            % ', '.join(AUDIO_EXT)
        )

    return (
        media,
        audio,
        script_text
    )


# -------------------------------------------------------------- the sync

def transcribe(
    path,
    total,
    progress
):
    """Every spoken word as (text, start, end).

    FFmpeg decodes the voiceover into a mono 16 kHz NumPy waveform first.

    Passing a NumPy waveform directly to faster-whisper bypasses its internal
    PyAV decoder. This prevents PyAV compatibility changes from breaking the
    transcription stage.

    CPU is intentional because free Colab runtimes can have CUDA, cuDNN and
    ctranslate2 combinations that are not compatible with each other.
    """

    import logging
    import warnings
    import numpy as np

    from faster_whisper import WhisperModel

    progress(
        0.25,
        desc='Listening to the voiceover...'
    )

    logging.getLogger(
        'huggingface_hub'
    ).setLevel(
        logging.ERROR
    )

    # Decode any supported input format to:
    #
    # mono
    # 16 kHz
    # float32 PCM
    #
    # stdout contains only the raw PCM bytes.
    p = subprocess.run(
        [
            'ffmpeg',
            '-v', 'error',
            '-i', path,
            '-f', 'f32le',
            '-acodec', 'pcm_f32le',
            '-ac', '1',
            '-ar', '16000',
            'pipe:1'
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    if p.returncode != 0:

        detail = p.stderr.decode(
            'utf8',
            'ignore'
        )[-500:]

        raise gr.Error(
            'The voiceover could not be decoded by ffmpeg. '
            + detail
        )

    audio = np.frombuffer(
        p.stdout,
        dtype=np.float32
    )

    if audio.size == 0:

        raise gr.Error(
            'The voiceover decoded to empty audio.'
        )

    with warnings.catch_warnings():

        warnings.simplefilter(
            'ignore'
        )

        model = WhisperModel(
            'base',
            device='cpu',
            compute_type='int8'
        )

        segs, _ = model.transcribe(
            audio,
            beam_size=1,
            word_timestamps=True
        )

        out = [
            (
                w.word,
                float(w.start),
                float(w.end)
            )
            for s in segs
            for w in s.words
        ]

    out.sort(
        key=lambda w: w[1]
    )

    return out


def starts_from_script(
    blocks,
    spoken,
    total,
    min_matched=0.34
):
    """Find where each script line was spoken."""

    sw = []
    bounds = []

    for b in blocks:

        bounds.append(
            len(sw)
        )

        sw.extend(
            bare_words(b)
        )

    bounds.append(
        len(sw)
    )

    tw = [
        bare_words(w)[0]
        if bare_words(w)
        else ''
        for w, _, _ in spoken
    ]

    pairs = {}

    for i, j, size in difflib.SequenceMatcher(
        None,
        sw,
        tw,
        autojunk=False
    ).get_matching_blocks():

        for k in range(size):

            pairs[
                i + k
            ] = (
                j + k
            )

    out = []
    missing = 0

    for b in range(
        len(blocks)
    ):

        s = bounds[b]
        e = bounds[b + 1]

        hit = [
            i
            for i in range(
                s,
                e
            )
            if i in pairs
        ]

        if (
            e > s
            and
            len(hit)
            >=
            min_matched * (e - s)
        ):

            out.append(
                spoken[
                    pairs[
                        hit[0]
                    ]
                ][1]
            )

        else:

            out.append(
                None
            )

            missing += 1

    out[0] = 0.0

    return (
        space_unmatched(
            out,
            total
        ),
        missing
    )


def space_unmatched(
    starts,
    total
):

    out = list(starts)

    known = [
        i
        for i, v in enumerate(out)
        if v is not None
    ]

    if not known:

        return [
            i * total / len(out)
            for i in range(
                len(out)
            )
        ]

    for i, v in enumerate(out):

        if v is not None:
            continue

        prev = max(
            [
                k
                for k in known
                if k < i
            ],
            default=-1
        )

        nxt = min(
            [
                k
                for k in known
                if k > i
            ],
            default=len(out)
        )

        a = (
            out[prev]
            if prev >= 0
            else 0.0
        )

        b = (
            out[nxt]
            if nxt < len(out)
            else total
        )

        out[i] = (
            a
            +
            (b - a)
            *
            (i - prev)
            /
            (nxt - prev)
        )

    return out


def to_durations(
    starts,
    total,
    min_shot=0.4
):

    fixed = [
        0.0
    ]

    for s in sorted(
        float(x)
        for x in starts
    )[1:]:

        fixed.append(
            max(
                s,
                fixed[-1] + min_shot
            )
        )

    fixed = [
        s
        for s in fixed
        if s < total - min_shot
    ] or [
        0.0
    ]

    return (
        fixed,
        [
            (
                fixed[i + 1]
                if i + 1 < len(fixed)
                else total
            )
            -
            fixed[i]
            for i in range(
                len(fixed)
            )
        ]
    )


def frame_counts(
    durations,
    total_seconds
):

    want = math.ceil(
        total_seconds * FPS
    )

    n = [
        max(
            1,
            int(
                round(
                    d * FPS
                )
            )
        )
        for d in durations
    ]

    short = (
        want - sum(n)
    )

    i = 0

    while (
        short != 0
        and
        i < len(n) * 64
    ):

        j = (
            i % len(n)
        )

        if short > 0:

            n[j] += 1
            short -= 1

        elif n[j] > 1:

            n[j] -= 1
            short += 1

        i += 1

    return n


# ---------------------------------------------------------- shot styling

def plan_dips(
    starts,
    spoken,
    total,
    floor=0.60,
    relative=1.6
):

    dips = [
        None
    ] * len(starts)

    if (
        not spoken
        or len(starts) < 2
    ):

        return dips

    ends = [
        w[2]
        for w in spoken
    ]

    begins = [
        w[1]
        for w in spoken
    ]

    gaps = []

    for i in range(
        len(starts) - 1
    ):

        t = starts[i + 1]

        prev = [
            e
            for e in ends
            if e <= t + 0.01
        ]

        nxt = [
            b
            for b in begins
            if b >= t - 0.01
        ]

        gaps.append(
            (
                nxt[0] - prev[-1]
                if prev and nxt
                else 0.0
            )
        )

    ordered = sorted(
        gaps
    )

    typical = ordered[
        len(ordered) // 2
    ]

    need = max(
        floor,
        typical * relative
    )

    for i, gap in enumerate(
        gaps
    ):

        if gap < need:
            continue

        room = (
            gap - 0.10
        )

        fade = min(
            0.35,
            room * 0.55
        )

        hold = min(
            0.25,
            room * 0.30
        )

        dips[i] = (
            fade,
            hold
        )

    return dips


def plan_zooms(
    counts,
    videos=None,
    seed=20260916
):

    import random

    rnd = random.Random(
        seed
    )

    out = []

    for i, n in enumerate(
        counts
    ):

        if n < int(
            FPS * 1.0
        ):

            out.append(
                None
            )

            continue

        if (
            videos
            and
            videos[i]
        ):

            out.append(
                None
            )

            continue

        out.append(
            (
                'in'
                if i % 2 == 0
                else 'out',
                rnd.uniform(
                    1.06,
                    1.12
                )
            )
        )

    return out


def zoom_filter(
    n,
    direction,
    zmax,
    W,
    H
):

    span = max(
        1,
        n - 1
    )

    t = (
        '(on/%d)'
        % span
    )

    ease = (
        '(%s*%s*(3-2*%s))'
        % (
            t,
            t,
            t
        )
    )

    amp = (
        zmax - 1.0
    )

    if direction == 'in':

        z = (
            '(1+%f*%s)'
            % (
                amp,
                ease
            )
        )

    else:

        z = (
            '(%f-%f*%s)'
            % (
                zmax,
                amp,
                ease
            )
        )

    hw = (
        '(%d/(2*%s))'
        % (
            W,
            z
        )
    )

    hh = (
        '(%d/(2*%s))'
        % (
            H,
            z
        )
    )

    cx = (
        '%d/2'
        % W
    )

    cy = (
        '%d/2'
        % H
    )

    return (
        "perspective="
        "eval=frame:"
        "sense=source:"
        "interpolation=cubic:"
        "x0='{cx}-{hw}':"
        "y0='{cy}-{hh}':"
        "x1='{cx}+{hw}':"
        "y1='{cy}-{hh}':"
        "x2='{cx}-{hw}':"
        "y2='{cy}+{hh}':"
        "x3='{cx}+{hw}':"
        "y3='{cy}+{hh}'"
    ).format(
        cx=cx,
        cy=cy,
        hw=hw,
        hh=hh
    )


# --------------------------------------------------------------- the job

def build(
    media,
    audio,
    script_text,
    script_file,
    size_label,
    zoom_on,
    dip_on,
    progress=gr.Progress()
):

    if not media:

        raise gr.Error(
            'Add some images or clips first.'
        )

    if not audio:

        raise gr.Error(
            'Add your voiceover first.'
        )

    t_start = time.time()

    W, H = SIZES[
        size_label
    ]

    shutil.rmtree(
        f'{WORK}/norm',
        ignore_errors=True
    )

    shutil.rmtree(
        f'{WORK}/clips',
        ignore_errors=True
    )

    os.makedirs(
        f'{WORK}/norm',
        exist_ok=True
    )

    os.makedirs(
        f'{WORK}/clips',
        exist_ok=True
    )

    paths = sorted(
        [
            f
            for f in media
            if f.lower().endswith(
                MEDIA_EXT
            )
        ],
        key=lambda p:
        natural_key(
            os.path.basename(p)
        )
    )

    if not paths:

        raise gr.Error(
            'None of those files are images or video clips (%s).'
            % ', '.join(MEDIA_EXT)
        )

    videos = [
        is_video(p)
        for p in paths
    ]

    text = (
        script_text
        or ''
    )

    if (
        script_file
        and
        not text.strip()
    ):

        text = read_script_file(
            script_file
        )

    lines, problem = script_vs_files(
        text,
        paths
    )

    n_vid = sum(
        videos
    )

    thing = (
        'image or clip'
        if n_vid
        else 'image'
    )

    things = (
        'images and clips'
        if n_vid
        else 'images'
    )

    if not lines:

        raise gr.Error(
            'Add your script - one line per %s, in the same order. Without '
            'it the tool can tell WHEN you paused but not WHICH shot '
            'belongs to which sentence.'
            % thing
        )

    if problem:

        raise gr.Error(
            problem
        )

    total = duration(
        audio
    )

    if (
        not has_sound(audio)
        or
        total < 1.0
    ):

        raise gr.Error(
            'That voiceover file has no sound in it that could be read. Use '
            'the recording itself (%s), not a link or a document.'
            % ', '.join(AUDIO_EXT)
        )

    progress(
        0.05,
        desc=(
            'Preparing the %s...'
            % things
        )
    )

    def normalise(
        i_path
    ):

        i, path = i_path

        if is_video(path):
            return path

        out = (
            f'{WORK}/norm/n_{i:05d}.jpg'
        )

        with Image.open(
            path
        ) as im:

            ImageOps.fit(
                im.convert('RGB'),
                (W, H),
                method=Image.LANCZOS
            ).save(
                out,
                quality=95
            )

        return out

    with ThreadPoolExecutor(
        max_workers=2
    ) as ex:

        frames = list(
            ex.map(
                normalise,
                enumerate(paths)
            )
        )

    spoken = transcribe(
        audio,
        total,
        progress
    )

    progress(
        0.45,
        desc='Matching the script to the voiceover...'
    )

    starts, missing = starts_from_script(
        lines,
        spoken,
        total
    )

    starts, durations = to_durations(
        starts,
        total
    )

    frames = frames[
        :len(starts)
    ]

    videos = videos[
        :len(starts)
    ]

    counts = frame_counts(
        durations,
        total
    )

    dips = (
        plan_dips(
            starts,
            spoken,
            total
        )
        if dip_on
        else [None] * len(frames)
    )

    zooms = (
        plan_zooms(
            counts,
            videos
        )
        if zoom_on
        else [None] * len(frames)
    )

    t0 = time.time()

    def build_clip(
        args
    ):

        i, path = args

        n = counts[i]

        out = (
            f'{WORK}/clips/c_{i:05d}.mp4'
        )

        if videos[i]:

            src = [
                '-stream_loop',
                '-1',
                '-i',
                path
            ]

            chain = [
                (
                    'scale=%d:%d:'
                    'force_original_aspect_ratio=increase'
                    % (
                        W,
                        H
                    )
                ),
                (
                    'crop=%d:%d'
                    % (
                        W,
                        H
                    )
                ),
                (
                    'fps=%d'
                    % FPS
                ),
                'setsar=1'
            ]

        else:

            src = [
                '-loop',
                '1',
                '-framerate',
                str(FPS),
                '-i',
                path
            ]

            chain = [
                (
                    'scale=%d:%d:flags=lanczos'
                    % (
                        W,
                        H
                    )
                )
            ]

            if zooms[i]:

                chain.append(
                    zoom_filter(
                        n,
                        zooms[i][0],
                        zooms[i][1],
                        W,
                        H
                    )
                )

            chain += [
                (
                    'fps=%d'
                    % FPS
                ),
                'setsar=1'
            ]

        if dips[i]:

            fade, hold = dips[i]

            dur = (
                n / FPS
            )

            chain.append(
                (
                    'fade=t=out:st=%.3f:d=%.3f'
                    % (
                        max(
                            0.0,
                            dur - hold - fade
                        ),
                        fade
                    )
                )
            )

        run(
            [
                'ffmpeg',
                '-y',
                '-loglevel',
                'error'
            ]
            +
            src
            +
            [
                '-vf',
                ','.join(chain),
                '-frames:v',
                str(n),
                '-c:v',
                'libx264',
                '-crf',
                '20',
                '-preset',
                'veryfast',
                '-pix_fmt',
                'yuv420p',
                '-an',
                out
            ]
        )

        return out

    clips = []

    with ThreadPoolExecutor(
        max_workers=2
    ) as ex:

        for k, clip in enumerate(
            ex.map(
                build_clip,
                list(
                    enumerate(frames)
                )
            )
        ):

            clips.append(
                clip
            )

            done = (
                k + 1
            )

            left = (
                len(frames)
                -
                (k + 1)
            )

            eta = ''

            if (
                done >= 2
                and
                left
            ):

                eta = (
                    '  -  about %s left'
                    % fmt_time(
                        (
                            time.time() - t0
                        )
                        /
                        done
                        *
                        left
                        +
                        8
                    )
                )

            progress(
                0.5
                +
                0.45
                *
                done
                /
                len(frames),
                desc=(
                    'Rendering shot %d of %d%s'
                    % (
                        done,
                        len(frames),
                        eta
                    )
                )
            )

    progress(
        0.96,
        desc='Joining and adding the voiceover...'
    )

    lst = (
        f'{WORK}/clips.txt'
    )

    with open(
        lst,
        'w'
    ) as f:

        for p in clips:

            f.write(
                "file '%s'\n"
                % p
            )

    out_path = (
        f'{WORK}/video.mp4'
    )

    run(
        [
            'ffmpeg',
            '-y',
            '-loglevel',
            'error',
            '-f',
            'concat',
            '-safe',
            '0',
            '-i',
            lst,
            '-i',
            audio,
            '-c:v',
            'copy',
            '-c:a',
            'aac',
            '-b:a',
            '192k',
            '-shortest',
            out_path
        ]
    )

    n_zoom = sum(
        1
        for z in zooms
        if z
    )

    n_dip = sum(
        1
        for d in dips
        if d
    )

    n_vid = sum(
        videos
    )

    chips = [
        (
            '%d shots'
            % len(frames),
            True
        ),
        (
            '%s'
            % fmt_time(total),
            False
        )
    ]

    if n_vid:

        chips.append(
            (
                '%d B-roll'
                % n_vid,
                False
            )
        )

    chips += [
        (
            '%d zooms'
            % n_zoom,
            False
        ),
        (
            '%d dips'
            % n_dip,
            False
        ),
        (
            'built in %s'
            % fmt_time(
                time.time()
                -
                t_start
            ),
            False
        )
    ]

    note = ''

    if missing:

        note = (
            '<div class="bt-note">'
            '%d of %d lines were not found in the '
            'voiceover and were placed between their neighbours. A few is '
            'normal. If it is most of them, the script is probably not the '
            'one that was read out.'
            '</div>'
            % (
                missing,
                len(lines)
            )
        )

    return (
        out_path,
        chip_html(chips)
        +
        note
    )


def fmt_time(s):

    return (
        '%d:%02d'
        % (
            int(s) // 60,
            int(s) % 60
        )
    )


EST_PER_SECOND = 0.9


def estimate_html(
    audio_path
):

    if not audio_path:
        return ''

    try:

        d = duration(
            audio_path
        )

    except Exception:

        return ''

    if (
        d < 1.0
        or
        not has_sound(
            audio_path
        )
    ):

        return ''

    secs = (
        d * EST_PER_SECOND
    )

    return (
        '<div class="bt-est">'
        'Voiceover <b>%s</b> '
        '&nbsp;&middot;&nbsp; '
        'estimated build time '
        '<b>%s to %s</b>'
        '</div>'
        % (
            fmt_time(d),
            fmt_time(
                secs * 0.6
            ),
            fmt_time(
                secs * 1.9
            )
        )
    )


def chip_html(
    chips
):

    return ''.join(
        (
            '<span class="bt-chip">%s%s</span>'
            % (
                (
                    '<span class="bt-dot"></span>'
                    if dot
                    else ''
                ),
                html.escape(
                    str(t)
                )
            )
        )
        for t, dot in chips
    )


def script_check(
    text,
    script_file,
    kept
):

    if (
        script_file
        and
        not (
            text or ''
        ).strip()
    ):

        try:

            text = read_script_file(
                script_file
            )

        except Exception:

            return ''

    paths = list(
        kept or []
    )

    lines, problem = script_vs_files(
        text,
        paths
    )

    if (
        not paths
        or
        not lines
    ):

        return ''

    if problem:

        return (
            '<div class="bt-note bt-note-warn">%s</div>'
            % html.escape(
                problem
            )
        )

    return (
        '<div class="bt-note bt-note-ok">'
        '%d lines, %d files - these line up.'
        '</div>'
        % (
            len(lines),
            len(paths)
        )
    )


def on_script_file(
    path
):

    if not path:

        return gr.update()

    try:

        return read_script_file(
            path
        )

    except Exception:

        raise gr.Error(
            'That script file could not be read. Use .txt or .docx.'
        )


# -------------------------------------------------- adding files in batches

def media_summary(
    kept,
    added=0,
    dup=0,
    skipped=0,
    clash=()
):

    if not kept:
        return ''

    names = [
        os.path.basename(p)
        for p in kept
    ]

    n_vid = sum(
        is_video(p)
        for p in kept
    )

    mb = (
        sum(
            os.path.getsize(p)
            for p in kept
            if os.path.exists(p)
        )
        /
        1e6
    )

    bits = [
        (
            '<b>%d</b> file%s'
            % (
                len(kept),
                ''
                if len(kept) == 1
                else 's'
            )
        )
    ]

    if n_vid:

        bits.append(
            (
                '%d image%s + %d clip%s'
                % (
                    len(kept) - n_vid,
                    ''
                    if len(kept) - n_vid == 1
                    else 's',
                    n_vid,
                    ''
                    if n_vid == 1
                    else 's'
                )
            )
        )

    bits.append(
        '%.1f MB'
        % mb
    )

    if added:

        bits.append(
            'just added %d'
            % added
        )

    safe = [
        html.escape(n)
        for n in names
    ]

    order = (
        ', '.join(
            safe[:3]
        )
        +
        ' &hellip; '
        +
        ', '.join(
            safe[-3:]
        )
        if len(safe) > 6
        else
        ', '.join(safe)
    )

    notes = []

    if clash:

        shown = ', '.join(
            html.escape(c)
            for c in sorted(
                clash
            )[:3]
        )

        notes.append(
            (
                '%d file%s kept out because %s already in the list, '
                'but the contents differ (%s%s). Rename them - two shots '
                'cannot share a filename.'
                % (
                    len(clash),
                    ''
                    if len(clash) == 1
                    else 's',
                    (
                        'that name is'
                        if len(clash) == 1
                        else 'those names are'
                    ),
                    shown,
                    ''
                    if len(clash) <= 3
                    else ', &hellip;'
                )
            )
        )

    same = (
        dup - len(clash)
    )

    if (
        same
        and
        not added
        and
        not clash
    ):

        notes.append(
            (
                'Nothing new to add - %s already in the list.'
                % (
                    'that file was'
                    if same == 1
                    else
                    'all %d of those files were'
                    % same
                )
            )
        )

    elif same:

        notes.append(
            '%d already in the list, so skipped.'
            % same
        )

    if skipped:

        notes.append(
            (
                '%d file%s skipped - not an image or video clip.'
                % (
                    skipped,
                    ''
                    if skipped == 1
                    else 's'
                )
            )
        )

    note = (
        '<div class="bt-note">%s</div>'
        % '<br>'.join(notes)
        if notes
        else ''
    )

    return (
        '<div class="bt-est">%s<br>'
        'Shot order: %s%s'
        '</div>'
        % (
            ' &nbsp;&middot;&nbsp; '.join(
                bits
            ),
            order,
            note
        )
    )


def add_media(
    new,
    kept
):

    kept = list(
        kept or []
    )

    have = {
        os.path.basename(p): p
        for p in kept
    }

    added = 0
    dup = 0
    skipped = 0

    clash = []

    for p in (
        new or []
    ):

        if not str(p).lower().endswith(
            MEDIA_EXT
        ):

            skipped += 1
            continue

        b = os.path.basename(
            p
        )

        if b in have:

            dup += 1

            try:

                if (
                    os.path.getsize(p)
                    !=
                    os.path.getsize(
                        have[b]
                    )
                ):

                    clash.append(
                        b
                    )

            except OSError:

                pass

            continue

        have[b] = p

        kept.append(
            p
        )

        added += 1

    kept.sort(
        key=lambda p:
        natural_key(
            os.path.basename(p)
        )
    )

    return (
        kept,
        media_summary(
            kept,
            added,
            dup,
            skipped,
            clash
        ),
        gr.update(
            value=None
        )
    )


def clear_media():

    return (
        [],
        '',
        gr.update(
            value=None
        )
    )


# ------------------------------------------------------- errors people can read

def error_html(
    msg,
    detail=''
):

    more = ''

    if detail:

        more = (
            '<details>'
            '<summary>technical detail</summary>'
            '<pre>%s</pre>'
            '</details>'
            % html.escape(
                detail.strip()
            )
        )

    return (
        '<div class="bt-err">'
        '<b>That did not build.</b><br>'
        '%s%s'
        '</div>'
        % (
            html.escape(
                msg
            ),
            more
        )
    )


def reason(e):

    return (
        getattr(
            e,
            'message',
            None
        )
        or
        str(e)
    )


def build_ui(
    media,
    audio,
    script_text,
    script_file,
    size_label,
    zoom_on,
    dip_on,
    progress=gr.Progress()
):

    try:

        out, stats = build(
            media,
            audio,
            script_text,
            script_file,
            size_label,
            zoom_on,
            dip_on,
            progress
        )

        return (
            out,
            stats,
            ''
        )

    except gr.Error as e:

        return (
            None,
            '',
            error_html(
                reason(e)
            )
        )

    except Exception as e:

        return (
            None,
            '',
            error_html(
                (
                    '%s: %s'
                    % (
                        type(e).__name__,
                        e
                    )
                ),
                traceback.format_exc()
            )
        )


def lock_go():

    return gr.update(
        interactive=False
    )


def unlock_go():

    return gr.update(
        interactive=True
    )


def drive_ui(
    link,
    progress=gr.Progress()
):

    try:

        media, audio, text = drive_pull(
            link,
            progress
        )

        return (
            media,
            audio,
            text,
            media,
            media_summary(
                media
            ),
            ''
        )

    except gr.Error as e:

        return (
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            error_html(
                reason(e)
            )
        )

    except Exception as e:

        return (
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            error_html(
                (
                    '%s: %s'
                    % (
                        type(e).__name__,
                        e
                    )
                ),
                traceback.format_exc()
            )
        )


# ------------------------------------------------------------------- UI

CSS = """
:root, .gradio-container {
  --bt-bg:#14161c;
  --bt-panel:#1c1f29;
  --bt-panel2:#232734;
  --bt-line:#2c3140;
  --bt-txt:#e7e9ee;
  --bt-muted:#8b93a7;
  --bt-accent:#7c5cff;
  --bt-accent2:#5b8cff;
  --bt-ok:#3ecf8e;
  --bt-warn:#f5b14c;
}

.gradio-container {
  background:var(--bt-bg) !important;
  color:var(--bt-txt) !important;
  font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif !important;
  font-size:13px !important;
  max-width:560px !important;
  margin:0 auto !important;
}

#bt-h1 h1 {
  color:var(--bt-accent) !important;
  -webkit-text-fill-color:var(--bt-accent) !important;
  display:inline-block !important;
  font-size:17px !important;
  font-weight:700 !important;
  letter-spacing:.3px !important;
  margin:0 0 3px !important;
}

#bt-h1 h1 span {
  color:var(--bt-muted) !important;
  -webkit-text-fill-color:var(--bt-muted) !important;
  font-size:11px !important;
  font-weight:500 !important;
  letter-spacing:.5px !important;
  margin-left:5px !important;
}

#bt-sub p {
  color:var(--bt-muted) !important;
  font-size:11px !important;
  margin:0 0 14px !important;
}

.bt-card {
  background:var(--bt-panel) !important;
  border:1px solid var(--bt-line) !important;
  border-radius:10px !important;
  padding:10px 11px !important;
  margin-bottom:10px !important;
}

.gradio-container .block {
  border-color:var(--bt-line) !important;
}

.bt-card .bt-card {
  border:none !important;
  background:transparent !important;
  padding:0 !important;
  margin:0 !important;
  border-radius:0 !important;
}

label[data-testid="block-label"],
span[data-testid="block-info"],
.bt-card > .label-wrap > span {
  font-size:10px !important;
  text-transform:uppercase !important;
  letter-spacing:.9px !important;
  color:var(--bt-muted) !important;
  font-weight:600 !important;
  background:transparent !important;
  border:none !important;
  box-shadow:none !important;
}

.bt-primary button,
button.bt-primary {
  background:linear-gradient(
      90deg,
      var(--bt-accent),
      var(--bt-accent2)
  ) !important;
  border:none !important;
  border-radius:8px !important;
  color:#fff !important;
  font-size:13px !important;
  font-weight:600 !important;
  padding:10px !important;
}

.bt-primary button:disabled,
button.bt-primary:disabled,
.bt-primary button[disabled],
button.bt-primary[disabled] {
  background:var(--bt-panel2) !important;
  color:var(--bt-muted) !important;
  cursor:not-allowed !important;
  opacity:1 !important;
}

.bt-slot:has(.prose:empty) {
  display:none !important;
}

.bt-slot,
.styler:has(> .bt-slot) {
  background:transparent !important;
  border:none !important;
  box-shadow:none !important;
}

.bt-chip {
  display:inline-flex;
  align-items:center;
  gap:5px;
  background:var(--bt-panel2);
  border:1px solid var(--bt-line);
  border-radius:20px;
  padding:3px 9px;
  font-size:11px;
  color:var(--bt-muted);
  margin:0 6px 6px 0;
}

.bt-dot {
  width:7px;
  height:7px;
  border-radius:50%;
  background:var(--bt-ok);
  display:inline-block;
}

.bt-note {
  color:var(--bt-warn);
  font-size:11px;
  line-height:1.5;
  margin-top:6px;
}

.bt-note-warn {
  color:var(--bt-warn);
}

.bt-note-ok {
  color:var(--bt-muted);
}

.bt-est {
  color:var(--bt-muted);
  font-size:11px;
  line-height:1.5;
  margin:0 0 8px 2px;
}

.bt-est b {
  color:var(--bt-txt);
  font-weight:600;
}

.bt-err {
  background:#2a1a1e !important;
  border:1px solid #7d3a44 !important;
  border-radius:10px !important;
  color:#ffc9d1 !important;
  font-size:12px !important;
  line-height:1.55 !important;
  padding:10px 12px !important;
  margin-bottom:10px !important;
}

.bt-err b {
  color:#fff !important;
}

.bt-err details {
  margin-top:7px;
}

.bt-err summary {
  cursor:pointer;
  color:var(--bt-muted);
  font-size:10.5px;
  text-transform:uppercase;
  letter-spacing:.8px;
}

.bt-err pre {
  white-space:pre-wrap;
  font-size:10px;
  line-height:1.45;
  color:var(--bt-muted);
  margin:6px 0 0;
  max-height:200px;
  overflow:auto;
}

#bt-hint p,
#bt-hint2 p {
  color:var(--bt-muted) !important;
  font-size:10.5px !important;
  line-height:1.5 !important;
  margin:7px 2px 0 !important;
}

#bt-hint code,
#bt-hint2 code {
  background:var(--bt-panel2) !important;
  border:1px solid var(--bt-line) !important;
  border-radius:4px !important;
  color:var(--bt-txt) !important;
  font-size:10px !important;
  padding:0 4px !important;
}

.bt-ghost button,
button.bt-ghost {
  background:var(--bt-panel2) !important;
  border:1px solid var(--bt-line) !important;
  border-radius:8px !important;
  color:var(--bt-txt) !important;
  font-size:12px !important;
  font-weight:500 !important;
  padding:7px !important;
}

button.bt-reset {
  flex:0 0 auto !important;
  width:auto !important;
  min-width:0 !important;
  align-self:flex-start !important;
  padding:4px 11px !important;
  font-size:10.5px !important;
  font-weight:400 !important;
  color:var(--bt-muted) !important;
}

button.bt-reset:hover {
  color:var(--bt-txt) !important;
}

.upload-container {
  height:auto !important;
}

.upload-container .wrap {
  min-height:0 !important;
  padding:16px 0 !important;
}

.file-preview,
.file-preview *,
table.file-preview td {
  background:var(--bt-panel2) !important;
  color:var(--bt-txt) !important;
  border-color:var(--bt-line) !important;
}

.gradio-container ::-webkit-scrollbar {
  width:9px;
  height:9px;
}

.gradio-container ::-webkit-scrollbar-track {
  background:var(--bt-panel);
}

.gradio-container ::-webkit-scrollbar-thumb {
  background:var(--bt-line);
  border-radius:5px;
}

textarea,
input[type="text"] {
  background:var(--bt-panel2) !important;
  color:var(--bt-txt) !important;
  border:1px solid var(--bt-line) !important;
  font-size:12px !important;
}

footer,
.built-with,
.show-api {
  display:none !important;
}
"""


THEME = gr.themes.Base(
    primary_hue=gr.themes.colors.purple,
    neutral_hue=gr.themes.colors.slate,
).set(
    body_background_fill='#14161c',
    background_fill_primary='#1c1f29',
    background_fill_secondary='#232734',
    border_color_primary='#2c3140',
    body_text_color='#e7e9ee',
    body_text_color_subdued='#8b93a7',
    block_background_fill='#1c1f29',
    block_border_color='#2c3140',
    block_radius='10px',
    input_background_fill='#232734',
)


with gr.Blocks(
    title='BuildTuber Auto Edit v2.0',
    analytics_enabled=False
) as demo:

    gr.Markdown(
        '# BuildTuber Auto Edit <span>v2.0</span>',
        elem_id='bt-h1'
    )

    gr.Markdown(
        'Upload your assets, choose your settings and generate your '
        'finished video.',
        elem_id='bt-sub'
    )

    with gr.Group(
        elem_classes='bt-card'
    ):

        imgs = gr.File(
            label='IMAGES & VIDEO CLIPS',
            file_count='multiple',
            file_types=[
                'image',
                'video'
            ]
        )

        kept = gr.State(
            []
        )

        media_info = gr.HTML()

        with gr.Row():

            clear_btn = gr.Button(
                'Clear Files',
                elem_classes=[
                    'bt-ghost',
                    'bt-reset'
                ]
            )

        gr.Markdown(
            'Files are used in **filename order**, so name them '
            'sequentially - for example `frame_001`, `frame_002`, '
            '`frame_003` - matching the order of the lines in your '
            'script.\n\n'
            'You can upload everything at once, or add them in '
            'multiple batches. For large projects, uploading in '
            'smaller batches may be more reliable.\n\n'
            'Video clips work like images: short clips repeat to fill '
            'the scene, while longer clips are trimmed.',
            elem_id='bt-hint'
        )

    with gr.Group(
        elem_classes='bt-card'
    ):

        aud = gr.Audio(
            label='VOICEOVER',
            type='filepath',
            waveform_options=gr.WaveformOptions(
                waveform_color='#7c5cff',
                waveform_progress_color='#5b8cff'
            )
        )

    with gr.Group(
        elem_classes='bt-card'
    ):

        script = gr.Textbox(
            label='SCRIPT  (one line per image or clip)',
            lines=6,
            max_lines=14,
            placeholder=(
                'Paste your script here, one line per '
                'image or clip.\n\n'
                'Or drop a .txt / .docx below.'
            )
        )

        script_f = gr.File(
            label='OR A .TXT / .DOCX FILE',
            file_types=[
                '.txt',
                '.docx'
            ]
        )

        script_info = gr.HTML(
            elem_classes='bt-slot'
        )

    with gr.Group(
        elem_classes='bt-card'
    ):

        size = gr.Dropdown(
            list(SIZES),
            value=list(SIZES)[0],
            label='OUTPUT SIZE'
        )

        zoom_on = gr.Checkbox(
            label='Ken Burns zoom',
            value=True
        )

        dip_on = gr.Checkbox(
            label='Dip to black at section breaks',
            value=True
        )

    with gr.Accordion(
        'Or load from a Google Drive folder',
        open=False,
        elem_classes='bt-card'
    ):

        drive_link = gr.Textbox(
            label='DRIVE FOLDER LINK',
            placeholder='https://drive.google.com/drive/folders/...'
        )

        drive_go = gr.Button(
            'Load from Drive',
            elem_classes='bt-ghost'
        )

        gr.Markdown(
            'Put the images, clips, voiceover and script in one '
            'folder. Needs "use_google_drive" ticked in Step 1.',
            elem_id='bt-hint2'
        )

    est = gr.HTML(
        elem_classes='bt-slot'
    )

    go = gr.Button(
        'Build Video',
        variant='primary',
        elem_classes='bt-primary'
    )

    err = gr.HTML(
        elem_classes='bt-slot'
    )

    stats = gr.HTML(
        elem_classes='bt-slot'
    )

    video = gr.Video(
        label='RESULT',
        interactive=False
    )

    script_f.change(
        on_script_file,
        script_f,
        script
    )

    aud.change(
        estimate_html,
        aud,
        est
    )

    _recheck = dict(
        fn=script_check,
        inputs=[
            script,
            script_f,
            kept
        ],
        outputs=script_info
    )

    script.change(
        **_recheck
    )

    script_f.change(
        **_recheck
    )

    kept.change(
        **_recheck
    )

    imgs.upload(
        add_media,
        [
            imgs,
            kept
        ],
        [
            kept,
            media_info,
            imgs
        ]
    )

    clear_btn.click(
        clear_media,
        None,
        [
            kept,
            media_info,
            imgs
        ]
    )

    drive_go.click(
        drive_ui,
        drive_link,
        [
            imgs,
            aud,
            script,
            kept,
            media_info,
            err
        ],
        show_progress_on=imgs
    )

    (
        go.click(
            lock_go,
            None,
            go,
            queue=False
        )
        .then(
            build_ui,
            [
                kept,
                aud,
                script,
                script_f,
                size,
                zoom_on,
                dip_on
            ],
            [
                video,
                stats,
                err
            ],
            show_progress_on=video
        )
        .then(
            unlock_go,
            None,
            go,
            queue=False
        )
    )


_kw = dict(
    inline=True,
    share=True,
    quiet=True,
    height=1100,
    show_error=True
)

if int(
    gr.__version__.split('.')[0]
) >= 6:

    _kw.update(
        theme=THEME,
        css=CSS
    )

else:

    demo.theme = THEME
    demo.css = CSS


demo.launch(
    **_kw
)

* Running on public URL: https://12b61f8fc330ca6156.gradio.live


---

### Want to automate the rest too?

BuildTuber Auto Edit handles the final editing step once you already have your
images, clips and voiceover.

BuildTuber Faceless Video Generator can also generate the scene prompts, images
and voiceover directly from your script.

Full 7 min Stickman videos for less than 50 cents.

Try BuildTuber →
https://app.buildtuber.com